<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第3章：评估推理模型

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "sympy",
    "tokenizers"  # 由 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<br>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F01_raschka.webp?1" width="700px">

&nbsp;
## 3.1 构建数学验证器

- 本节没有代码。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F02_raschka.webp?1" width="700px">

<br>
<br>
<br>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F03_raschka.webp?1" width="700px">

&nbsp;
## 3.2 加载预训练模型进行文本生成

- 本节会再次加载第2章中要评估的那个模型。

In [ ]:
from pathlib import Path
import torch


from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)


def load_model_and_tokenizer(
    which_model, device, use_compile, local_dir="qwen3"
):
    if which_model == "base":

        download_qwen3_small(
            kind="base", tokenizer_only=False, out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-base.json"
        model_path = Path(local_dir) / "qwen3-0.6B-base.pth"
        tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

    elif which_model == "reasoning":

        download_qwen3_small(
            kind="reasoning", tokenizer_only=False, out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-reasoning.json"
        model_path = Path(local_dir) / "qwen3-0.6B-reasoning.pth"
        tokenizer = Qwen3Tokenizer(
            tokenizer_file_path=tokenizer_path,
            apply_chat_template=True,
            add_generation_prompt=True,
            add_thinking=True,
        )

    else:
        raise ValueError(f"Invalid choice: which_model={which_model}")

    model = Qwen3Model(QWEN_CONFIG_06_B)
    model.load_state_dict(torch.load(model_path))

    model.to(device)

    if use_compile:
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        model = torch.compile(model)

    return model, tokenizer

- 这里我们使用基础版模型；完成本章后，你可以把`which_model="base"` 改成 `which_model="reasoning"` 再跑一次 notebook，以评估已经训练好的推理模型。

In [ ]:
from reasoning_from_scratch.ch02 import (
    get_device
)

WHICH_MODEL = "base"
device = get_device()

# 如果遇到兼容性问题，请尝试
# 取消注释下一行并重新运行 notebook
# device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model=WHICH_MODEL,
    device=device,
    use_compile=False
)

In [ ]:
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache
)

prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)

# 与第 2 章练习解答类似：
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)

all_token_ids = []
for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=2048,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0)
    decoded_id = tokenizer.decode(token_id.tolist())
    print(
        decoded_id,
        end="",
        flush=True
    )
    all_token_ids.append(token_id)

all_tokens = tokenizer.decode(all_token_ids)

- 如果你不熟悉 LaTeX 语法，上面的回答会很难读。
- 可以像下面这样用`Latex` 类把 LaTeX 渲染成更易读的格式。

In [ ]:
from IPython.display import Latex, display

display(Latex(all_tokens))

- 若只需渲染某些数学表达式，也可以使用 `Math` 类:

In [ ]:
from IPython.display import Math

display(Math(r"\dfrac{14}{3}"))

&nbsp;
## 3.3 编写封装以简化文本生成

- 上文我们已经加载了预训练 LLM，并搭建好了文本生成功能（如下图所示），它们正是本章余下内容将覆盖的评估流程前两步。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F05_raschka.webp?1" width="700px">

- 为了更方便，我们再封装一层文本生成函数，调用时只需传入模型、tokenizer、提示语和相关设置。

In [ ]:
def generate_text_stream_concat(
    model, tokenizer, prompt, device, max_new_tokens,
    verbose=False,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt), device=device
        ).unsqueeze(0)

    generated_ids = []
    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id.item())

        if verbose:
            print(
                tokenizer.decode(next_token_id.tolist()),
                end="",
                flush=True
            )
    return tokenizer.decode(generated_ids)


skip_portion = False

if not skip_portion:
    generated_text = generate_text_stream_concat(
        model, tokenizer, prompt, device,
        max_new_tokens=2048,
        verbose=True
    )

&nbsp;
## 3.4 提取最终答案框

- 本节先提取答案框（第 3 步）；下一节会把提取出的答案做标准化处理（第 4 步）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F06_raschka.webp?1" width="700px">

In [ ]:
model_answer = (
r"""... some explanation...
**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]
""")

In [ ]:
def get_last_boxed(text):
    # 查找最后一次出现的 "\boxed"
    boxed_start_idx = text.rfind(r"\boxed")
    if boxed_start_idx == -1:
        return None

    # 获取 "\boxed" 之后的位置
    current_idx = boxed_start_idx + len(r"\boxed")

    # 跳过 "\boxed" 之后的所有空白
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # 预期出现左花括号 "{"
    if current_idx >= len(text) or text[current_idx] != "{":
        return None

    # 解析可能嵌套的花括号
    current_idx += 1
    brace_depth = 1
    content_start_idx = current_idx

    while current_idx < len(text) and brace_depth > 0:
        char = text[current_idx]
        if char == "{":
            brace_depth += 1
        elif char == "}":
            brace_depth -= 1
        current_idx += 1

    # 处理花括号不平衡的情况
    if brace_depth != 0:
        return None

    # 提取最外层花括号中的内容
    return text[content_start_idx:current_idx-1]

In [ ]:
extracted_answer = get_last_boxed(model_answer)
print(extracted_answer)

In [ ]:
import re

RE_NUMBER = re.compile(
    r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"
)

def extract_final_candidate(text, fallback="number_then_full"):
    # 没有匹配项时的默认返回值
    result = ""

    if text:
        # 如果存在方框表达式，优先使用最后一个
        boxed = get_last_boxed(text.strip())
        if boxed:
            result = boxed.strip().strip("$ ")

        # 如果没有方框表达式，则尝试回退方案
        elif fallback in ("number_then_full", "number_only"):
            m = RE_NUMBER.findall(text)
            if m:
                # 使用最后一个数字
                result = m[-1]
            elif fallback == "number_then_full":
                # 如果未找到数字，则返回完整文本
                result = text
    return result

- 当找不到 boxed 内容时的后备策略：
    - "number_then_full": 优先提取最后一个简单数字，否则返回整段文本。
    - "number_only": 只提取最后一个简单数字，若没有则返回空字符串 `""`
    - "none": 仅提取 boxed 内容，若未找到则返回空字符串 `""`

In [ ]:
print(extract_final_candidate(model_answer))

In [ ]:
print(extract_final_candidate(r"\boxed{ 14/3. }"))

In [ ]:
print(extract_final_candidate("abc < > 14/3 abc"))

In [ ]:
print(extract_final_candidate("Text without numbers"))

&nbsp;
## 3.5 标准化提取出的答案

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F07_raschka.webp?1" width="700px">

- 上一节完成了答案提取（步骤 3），现在我们要对其做标准化处理（对应上一张图的步骤 4）。

In [ ]:
LATEX_FIXES = [  # 要替换的 LaTeX 格式
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"\\cdot", "*"),
    (r"\u00B7|\u00D7", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    (r"°", ""),
]

RE_SPECIAL = re.compile(r"<\|[^>]+?\|>")  # 移除 <|assistant|> 等聊天特殊词元

def normalize_text(text):
    if not text:
        return ""
    text = RE_SPECIAL.sub("", text).strip()
    SUPERSCRIPT_MAP = {
        "⁰": "0", "¹": "1", "²": "2", "³": "3", "⁴": "4",
        "⁵": "5", "⁶": "6", "⁷": "7", "⁸": "8", "⁹": "9",
        "⁺": "+", "⁻": "-", "⁽": "(", "⁾": ")",
    }

    # 移除开头的多项选择标签
    # 例如，将 "c. 3" -> 3，或 "b: 2" -> 2
    match = re.match(r"^[A-Za-z]\s*[.:]\s*(.+)$", text)
    if match:
        text = match.group(1)
        
    # 移除角度标记
    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text)   # ^{\circ}
    text = re.sub(r"\^\s*\\circ", "", text)             # ^\circ
    text = text.replace("°", "")                        # Unicode 度数符号

    # 如果整个字符串由 \text{...} 包裹，则移除外层包装
    match = re.match(r"^\\text\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x")

    # 移除行内/显示数学公式包装符 \( \) \[ \]
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)

    # 轻量 LaTeX 规范化
    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    def convert_superscripts(s, base=None):
        converted = "".join(
            SUPERSCRIPT_MAP[ch] if ch in SUPERSCRIPT_MAP else ch
            for ch in s
        )
        if base is None:
            return converted
        return f"{base}**{converted}"

    # 将 Unicode 上标转换为指数形式（例如 2² -> 2**2）
    text = re.sub(
        r"([0-9A-Za-z\)\]\}])([⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻]+)",
        lambda m: convert_superscripts(m.group(2), base=m.group(1)),
        text,
    )
    text = convert_superscripts(text)
    
    # 数字/根式
    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    text = re.sub(
        r"\\sqrt\s*\{([^}]*)\}",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )
    text = re.sub(
        r"\\sqrt\s+([^\\\s{}]+)",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )

    # 分数
    text = re.sub(
        r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )
    text = re.sub(
        r"\\frac\s+([^\s{}]+)\s+([^\s{}]+)",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )

    # 指数和带分数
    text = text.replace("^", "**")
    text = re.sub(
        r"(?<=\d)\s+(\d+/\d+)",
        lambda match: "+" + match.group(1),
        text,
    )

    # 1,234 -> 1234
    text = re.sub(
        r"(?<=\d),(?=\d\d\d(\D|$))",
        "",
        text,
    )

    return text.replace("{", "").replace("}", "").strip().lower()

In [ ]:
print(normalize_text(extract_final_candidate(model_answer)))

In [ ]:
print(normalize_text(r"$\dfrac{14}{3.}$"))

In [ ]:
print(normalize_text(r"\text{\[\frac{14}{3}\]}"))

In [ ]:
print(normalize_text("4/3"))

&nbsp;
## 3.6 验证数学等价性

- 本节实现基础功能：比较模型生成并提取出的答案与数据集中提供的正确答案（ground truth）是否等价，对应流程的第 5 步。
- 下一节（第 6 步）会进一步增强判分逻辑，让答案正确性评估更稳健。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F08_raschka.webp?1" width="700px">

In [ ]:
from sympy.parsing import sympy_parser as spp
from sympy.core.sympify import SympifyError
from tokenize import TokenError
from sympy.polys.polyerrors import PolynomialError

def sympy_parser(expr):
    # 避免因过长的无效回复而崩溃 
    # 某些训练不佳的模型（第 6 章）可能会输出这种内容
    if expr is None or len(expr) > 2000:
        return None
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                # 处理括号等标准转换
                *spp.standard_transformations,

                # 允许省略乘号（例如 "2x" -> 2*x）
                spp.implicit_multiplication_application,
            ),

            # 解析时求值，使简单常量得到化简（例如 2+3 -> 5）
            evaluate=True,
        )
    except (SympifyError, SyntaxError, TypeError, AttributeError,
            IndexError, TokenError, ValueError, PolynomialError):
        return None

- 看起来这里的错误处理很多，但这些都是我在评估 500 道 MATH-500 题目时遇到的真实问题，因为模型输出并不总是格式完美。

In [ ]:
print(sympy_parser(normalize_text(
    extract_final_candidate(model_answer)
)))

In [ ]:
print(sympy_parser("28/6"))

In [ ]:
from sympy import simplify

def equality_check(expr_gtruth, expr_pred):
    # 首先检查两个表达式是否为完全相同的字符串
    if expr_gtruth == expr_pred:
        return True

    # 将两个表达式解析为 SymPy 对象（解析失败时返回 None）
    gtruth, pred = sympy_parser(expr_gtruth), sympy_parser(expr_pred)

    # 如果两个表达式都成功解析，则尝试符号比较
    if gtruth is not None and pred is not None:
        try:
            # 如果差为 0，则二者等价
            return simplify(gtruth - pred) == 0
        except (SympifyError, TypeError):
            pass

    return False

In [ ]:
print(equality_check(
    normalize_text("13/4."),
    normalize_text(r"(13)/(4)")
))

In [ ]:
print(equality_check(
    normalize_text("0.5"),
    normalize_text(r"(1)/(2)")
))

In [ ]:
print(equality_check(
    normalize_text("14/3"),
    normalize_text("15/3")
))

In [ ]:
print(equality_check(
    normalize_text("(14/3, 2/3)"),
    normalize_text("(14/3, 4/6)")
))

&nbsp;
## 3.7 对答案评分

In [ ]:
def split_into_parts(text):
    result = [text]

    if text:
        # 检查文本是否类似元组或列表，例如 "(a, b)" 或 "[a, b]"
        if (
            len(text) >= 2
            and text[0] in "([" and text[-1] in ")]"
            and "," in text[1:-1]
        ):
            # 按括号内的逗号拆分并去除空白
            items = [p.strip() for p in text[1:-1].split(",")]
            if all(items):
                result = items
    else:
        # 如果文本为空，则返回空列表
        result = []

    return result

In [ ]:
split_into_parts(normalize_text(r"(14/3, 2/3)"))

In [ ]:
def grade_answer(pred_text, gt_text):
    result = False  # 检查失败时的默认结果

    # 仅当两个输入都是非空字符串时继续
    if pred_text is not None and gt_text is not None:
        gt_parts = split_into_parts(
            normalize_text(gt_text)
        )  # 将真实答案拆分为可比较的部分

        pred_parts = split_into_parts(
            normalize_text(pred_text)
        )  # 将预测拆分为可比较的部分

        # 确保两侧有效部分的数量相同
        if (gt_parts and pred_parts
           and len(gt_parts) == len(pred_parts)):
            result = all(
                equality_check(gt, pred)
                for gt, pred in zip(gt_parts, pred_parts)
            )  # 检查每个部分在数学上是否等价

    return result  # 仅当所有检查都通过时为 True

In [ ]:
grade_answer("14/3", r"\frac{14}{3}")

In [ ]:
grade_answer(r"(14/3, 2/3)", "(14/3, 4/6)")

In [ ]:
# 定义测试用例：（名称、预测、真实答案、预期结果）
tests = [
        ("check_1", "3/4", r"\frac{3}{4}", True),
        ("check_2", "(3)/(4)", r"3/4", True),
        ("check_3", r"\frac{\sqrt{8}}{2}", "sqrt(2)", True),
        ("check_4", r"\( \frac{1}{2} + \frac{1}{6} \)", "2/3", True),
        ("check_5", "(1, 2)", r"(1,2)", True),
        ("check_6", "(2, 1)", "(1, 2)", False),
        ("check_7", "(1, 2, 3)", "(1, 2)", False),
        ("check_8", "0.5", "1/2", True),
        ("check_9", "0.3333333333", "1/3", False),
        ("check_10", "1,234/2", "617", True),
        ("check_11", r"\text{2/3}", "2/3", True),
        ("check_12", "50%", "1/2", False),
        ("check_13", r"2\cdot 3/4", "3/2", True),
        ("check_14", r"90^\circ", "90", True),
        ("check_15", r"\left(\frac{3}{4}\right)", "3/4", True),
        ("check_16", r"2²", "2**2", True),
    ]


def run_demos_table(tests):
    header = ("Test", "Expect", "Got", "Status")
    rows = []
    for name, pred, gtruth, expect in tests:
        got = grade_answer(pred, gtruth)  # 运行等价性检查
        status = "PASS" if got == expect else "FAIL"
        rows.append((name, str(expect), str(got), status))

    data = [header] + rows
    
    # 计算每列最大宽度，使表格整齐对齐
    col_widths = [
        max(len(row[i]) for row in data)
        for i in range(len(header))
    ]

    # 逐行输出表格
    for row in data:
        line = " | ".join(
            row[i].ljust(col_widths[i])
            for i in range(len(header))
        )
        print(line)

    # 输出测试通过情况汇总
    passed = sum(r[3] == "PASS" for r in rows)
    print(f"\nPassed {passed}/{len(rows)}")

In [ ]:
run_demos_table(tests)

&nbsp;
## 3.8 加载评测数据集

- 上一节实现了基础的评估流水线。
- 本节加载要评估的数据集（步骤 7），下一节会在该数据集上应用这套流程来评估模型（步骤 8）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F09_raschka.webp?1" width="700px">

- 该数据集是通过下述脚本从 [HuggingFaceH4/MATH-500](https://huggingface.co/datasets/HuggingFaceH4/MATH-500)  仓库下载并预处理的，该脚本依赖  [`datasets`](https://huggingface.co/docs/datasets/en/index) 包（无需实际运行，仅供参考）：

```python
from datasets import load_dataset
import json

dset = load_dataset("HuggingFaceH4/MATH-500", split="test")

math_data = dset.to_list()
with open("math500_test.json", "w", encoding="utf-8") as f:
    json.dump(math_data, f, ensure_ascii=False, indent=2)
```

In [ ]:
import json
import requests

def load_math500_test(local_path="math500_test.json", save_copy=True):
    local_path = Path(local_path)
    url = (
        "https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/"
        "main/ch03/01_main-chapter-code/math500_test.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        if save_copy:  # 保存本地副本
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

math_data = load_math500_test()
print("Number of entries:", len(math_data))

In [ ]:
from pprint import pprint
pprint(math_data[0])

&nbsp;
## 3.9 评估模型

- 上一节已经加载了数据集；现在可以把前面搭好的评估流程应用到该数据集上，对模型进行评估（对应步骤 7）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F10_raschka.webp?1" width="700px">

In [ ]:
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Answer the question and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template

In [ ]:
prompt = (  # 与本章开头使用的提示词相同
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)
prompt_fmt = render_prompt(prompt)
print(prompt_fmt)

In [ ]:
generated_text = generate_text_stream_concat(
    model, tokenizer, prompt_fmt, device,
    max_new_tokens=2048,
    verbose=True
)

In [ ]:
# 以下是另一种提示词模板
# 将 "Question" 替换为 "Problem"

"""
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Problem:\n{prompt}\n\nAnswer:"
    )
    return template
"""

# 这会显著影响 MATH-500 结果：
# mps 上的基础模型：准确率从 20% 提升至 40%
# mps 上的推理模型：准确率从 90% 降至 60%

In [ ]:
# 也可以不使用提示词模板

"""
def render_prompt(prompt):
    return prompt
"""

# 这会显著影响 MATH-500 结果：
# mps 上的基础模型：准确率从 20% 提升至 70%
# mps 上的推理模型：准确率从 90% 降至 50%

In [ ]:
def mini_eval_demo(model, tokenizer, device):
    ex = {  # 使用 "problem" 和 "answer" 字段的测试样本
        "problem": "Compute 1/2 + 1/6.",
        "answer": "2/3"
    }
    prompt = render_prompt(ex["problem"])     # 1. 应用提示词模板
    gen_text = generate_text_stream_concat(   # 2. 生成回复
        model, tokenizer, prompt, device,
        max_new_tokens=64,
    )
    pred_answer = extract_final_candidate(gen_text)  # 3. 提取并归一化答案
    is_correct = grade_answer(                       # 4. 评判答案
        pred_answer, ex["answer"]
    )
    print(f"Device: {device}")
    print(f"Prediction: {pred_answer}")
    print(f"Ground truth: {ex['answer']}")
    print(f"Correct: {is_correct}")

In [ ]:
mini_eval_demo(model, tokenizer, device)

In [ ]:
import time


# 计算剩余时间的辅助函数
def eta_progress_message(
    processed,
    total,
    start_time,
    show_eta=False,
    label="Progress",
):
    progress = f"{label}: {processed}/{total}"
    pad_width = len(f"{label}: {total}/{total} | ETA: 00h 00m 00s")
    if not show_eta or processed <= 0:
        return progress.ljust(pad_width)

    elapsed = time.time() - start_time
    if elapsed <= 0:
        return progress.ljust(pad_width)

    remaining = max(total - processed, 0)

    if processed:
        avg_time = elapsed / processed
        eta_seconds = avg_time * remaining
    else:
        eta_seconds = 0

    eta_seconds = max(int(round(eta_seconds)), 0)
    minutes, rem_seconds = divmod(eta_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        eta = f"{hours}h {minutes:02d}m {rem_seconds:02d}s"
    elif minutes:
        eta = f"{minutes:02d}m {rem_seconds:02d}s"
    else:
        eta = f"{rem_seconds:02d}s"

    message = f"{progress} | ETA: {eta}"
    return message.ljust(pad_width)

def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")  # 使文件名兼容 Windows
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:  # 保存结果以供检查
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])    # 1. 应用提示词模板
            gen_text = generate_text_stream_concat(   # 2. 生成回复
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )

            extracted = extract_final_candidate(  # 3. 提取并归一化答案
                gen_text
            )
            is_correct = grade_answer(            # 4. 评判答案
                extracted, row["answer"]
            )
            num_correct += int(is_correct)

            record = {  # 保存记录以供检查
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:  # 在生成过程中输出回复
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"当前正确数： {num_correct}\n{'-'*50}"
                )

    # 输出汇总信息
    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"总耗时： {seconds_elapsed/60:.1f} min")
    print(f"日志已写入： {out_path}")
    return num_correct, num_examples, acc

- 为了演示并控制运行时间，这里只评估 10 个样例。

In [ ]:
print("Model:", WHICH_MODEL)
print("Device:", device)
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device, 
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)

| 模式      | 设备 | 准确率 | MATH-500 规模 | 耗时                  |
|-----------|--------|----------|---------------|-----------------------|
| 基础      | CPU    | 30%      | 10            | 0.7 分钟 (Mac Mini M4) |
| 基础      | MPS    | 20%      | 10            | 0.4 分钟 (Mac Mini M4) |
| 基础      | CUDA   | 30%      | 10            | 0.2 分钟 (DGX Spark)   |
| 基础      | XPU    | 30%      | 10            | 1.2 分钟 (Intel)       |
| 推理 | CPU    | 90%      | 10            | 9.5 分钟 (Mac Mini M4) |
| 推理 | MPS    | 80%      | 10            | 3.8 分钟 (Mac Mini M4) |
| 推理 | CUDA   | 90%      | 10            | 3.7 分钟 (DGX Spark)   |
| 推理 | XPU    | 70%      | 10            | 8.5 分钟 (Intel)       |


| 模式      | 设备 | 准确率 | MATH-500 规模    | 耗时                   |
|-----------|--------|----------|------------------|------------------------|
| 基础      | CUDA   | 15.6%    | 500              | 10.0 分钟 (DGX Spark)   |
| 推理 | CUDA   | 50.8%    | 500              | 182.2 分钟 (DGX Spark)  |

- 请注意，这些数值是在 PyTorch 2.8 上得到的，换成其它版本可能会有所不同。

- 上表列出了不同配置下的准确率。
- 这里的 “GPU” 指 NVIDIA（cuda）显卡；MPS 指 Apple Silicon M4。
- 推理版模型更慢，是因为它输出的文本更长。
- 虽然 Qwen3-Base 是基础预训练模型，且官方建议不用 chat 模板，但把 `tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)` 改成 `tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path, apply_chat_template=True)` 能显著提升 MATH-500 表现（可达 80%）。需要指出，MATH-500 测试集是否被用于训练尚不明确；在 LLM 时代，通常假定互联网上的数据都有可能进入训练集(参见 [here](https://github.com/rasbt/LLMs-from-scratch/pull/828#issuecomment-3324829736))
- 要跑完整 500 道 MATH-500 题，只需把 `evaluate_math500_stream` 中的 `math_data=math_data[:10],`  改成 `math_data=math_data,`
- T附加资料中包含批处理版脚本，可大幅提升评估吞吐(见 [../02_math500-verifier-scripts/README.md](../02_math500-verifier-scripts/README.md))在 H100 上，batch size=128 时，基础模型 3.3 分钟就能评完、推理版大约 14.6 分钟。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F11_raschka.webp?1" width="700px">

- 为方便起见，可以使用 [../02_math500-verifier-scripts/evaluate_math500.py](../02_math500-verifier-scripts/evaluate_math500.py) 脚本，从命令行以独立脚本形式运行 MATH-500 评估代码；更多用法请参阅 [../02_math500-verifier-scripts/README.md](../02_math500-verifier-scripts/README.md)
- [../02_math500-verifier-scripts/evaluate_math500_batched.py](../02_math500-verifier-scripts/evaluate_math500_batched.py) 脚本会以批处理模式运行本章代码
  - 这意味着每次前向传播会处理多个样本，从而加快评估，但需要更多内存
  - 在 H100 GPU 上使用 128 的批大小评估全部 500 个样本时，基础模型的运行时间会从 13.3 分钟降至 3.3 分钟
  - 同样，推理模型在这 500 个样本上的运行时间会从 185.4 分钟降至 14.6 分钟
  - H100 只是示例；该脚本也兼容其他 GPU（以及 CPU）


&nbsp;
## 总结

- 本节没有代码。